## Prerequisites

Before running this notebook, generate and save the robot mounting graph **once**:

```bash
python generate_mounting_graph.py
```

This is an interactive step — the Open3D window lets you crop unwanted surface regions. The result is saved to `data/mounting_graph.pkl` and loaded automatically when the notebook imports `config.params`.

# Import

In [ ]:
%load_ext autoreload
%autoreload 2

import random

from deap import base, creator, tools

import config.params as params
from config.params import Sensor, Gene, Individual, Population
import config.seeding as seeding
import config.sensors as sensors

# Initialization

In [ ]:
# Define the fitness to maximize
creator.create("FitnessMax", base.Fitness, weights=(1.0,))

# Define the Individual class as a list of Gene objects with an associated fitness.
creator.create("Individual", list, fitness=creator.FitnessMax)  # type: ignore

toolbox = base.Toolbox()

In [ ]:
import custom_toolbox.initialize.initialize as initialize

# Register the individual generator.
toolbox.register("individual", initialize.create_individual, creator.Individual)    # type: ignore

# Register the population generator (creates a list of individuals).
toolbox.register("population", tools.initRepeat, list, toolbox.individual)  # type: ignore

# NOTE: Consider Demes
# A deme is a sub-population that is contained in a population.
# https://deap.readthedocs.io/en/master/tutorials/basic/part1.html#demes

# Create initial population.
population = initialize.create_seeded_population(
    creator.Individual, # type: ignore
    toolbox.individual, # type: ignore
    seeding.SEED_INDIVIDUALS,
    params.POPULATION_SIZE,
)

In [ ]:
import utils.visualization as visualization

visualization.visualize_population(population, max_display=10)

# Evaluation

In [ ]:
import custom_toolbox.evaluate.evaluate_fitness_raycast as evaluate

# Build the ray-cast coverage evaluator once (loads chassis mesh + raycasting scene).
evaluator = evaluate.CoverageEvaluator()

# Register the custom evaluation function (use the standard 'evaluate' name).
toolbox.register("evaluate", evaluator.evaluate_individual)

# Selection

In [ ]:
# Tournament selection with tournament size of 3.
toolbox.register("select", tools.selTournament, tournsize=3)

# Mutation

In [ ]:
import custom_toolbox.mutate.mutate as mutate

# Register the custom mutation function.
toolbox.register("mutate", mutate.mutate_sensor_layout)

# Mating

In [ ]:
import custom_toolbox.mate.mate as mate

# Register the custom crossover function.
toolbox.register("mate", mate.cx_variable_length_bounded)

# Evolution

In [ ]:
import numpy as np
from deap import tools

# 1. Create a Statistics object that looks at the first objective values
stats = tools.Statistics(key=lambda ind: ind.fitness.values[0])

# 2. Register the metrics you want to track
stats.register("avg", np.mean)
stats.register("std", np.std)
stats.register("min", np.min)
stats.register("max", np.max)

# 3. Create a Logbook to store the data nicely
logbook = tools.Logbook()
logbook.header = ["gen", "nevals"] + (stats.fields if stats else [])

In [ ]:
from utils.ga_run import run_evolution
USE_LENGTH_NICHING = True

# Single run, driven by the USE_LENGTH_NICHING toggle set in the Selection cell.
# (To compare both strategies head to head, see the "Compare selection
# strategies" section below.)
result = run_evolution(
    population,
    toolbox,
    use_length_niching=USE_LENGTH_NICHING,
    seed=42,
)

# Unpack into the names the visualization cells below expect.
logbook = result.logbook
hof = result.hof
best_per_length = result.best_per_length
per_length_evolution = result.per_length_evolution
population[:] = result.population

print("Best overall layout found:", hof[0])
print("Number of sensors in best layout:", len(hof[0]))
print("Best overall fitness:", hof[0].fitness.values)

print("\n-- Best layout per sensor count --")
for length in best_per_length.lengths:
    best = best_per_length[length]
    print(f"  {length} sensor(s): fitness={best.fitness.values[0]:.4f}")


In [ ]:
import utils.visualization as visualization

visualization.visualize_evolution(logbook)

In [ ]:
visualization.visualize_evolution_per_length(per_length_evolution)

In [ ]:
visualization.visualize_length_distribution(per_length_evolution)

In [ ]:
visualization.visualize_coverage_maps(hof[0], evaluator=evaluator)

In [ ]:
visualization.visualize_best_layout(hof[0], evaluator=evaluator)

# Best Layout per Sensor Count

`best_per_length` holds the fittest individual found at each sensor count (1..`MAX_SENSORS_PER_INDIVIDUAL`). The comparison view shows the fitness/cost tradeoff across counts; the detail view renders each champion's coverage maps and 3D layout.

In [ ]:
visualization.visualize_length_comparison(best_per_length, evaluator=evaluator)

In [ ]:
visualization.visualize_bests_per_length(best_per_length, evaluator=evaluator)

# Compare selection strategies

Run **tournament + elitism** and **length niching** from the same initial population and seed, then compare them. The only controlled difference is the survivor-selection operator, so any divergence is attributable to it.

- **Fitness over generations** — does either strategy reach a better overall layout?
- **Population composition** — length niching should keep every sensor-count band alive; tournament tends to collapse toward one count.
- **Best per sensor count** — which strategy finds the better layout at each count? (Niching's per-length niches should dominate the minority counts.)
- **Per-count evolution** — small multiples; a broken line means that count went extinct under that strategy.

In [ ]:
import config.seeding as seeding
import custom_toolbox.initialize.initialize as initialize
from utils.ga_run import run_evolution
import utils.comparison as comparison

# Build ONE fresh starting population and run both strategies from it with the
# same seed, so the only difference between the runs is the selection operator.
# (Tip: lower params.NGEN / params.POPULATION_SIZE for a quick A/B; each run is
# a full evolution. The shared evaluator cache makes the second run cheaper.)
initial_population = initialize.create_seeded_population(
    creator.Individual,  # type: ignore
    toolbox.individual,  # type: ignore
    seeding.SEED_INDIVIDUALS,
    params.POPULATION_SIZE,
)

COMPARE_SEED = 42

results = {}
for use_niching in (True, False):
    res = run_evolution(
        initial_population,
        toolbox,
        use_length_niching=use_niching,
        seed=COMPARE_SEED,
    )
    results[res.label] = res

In [ ]:
# Overall best (solid) and average (dashed) fitness per strategy.
comparison.compare_evolution(results)

In [ ]:
# Population composition over generations, one panel per strategy. This is the
# headline contrast: length niching holds every sensor-count band open, while
# tournament + elitism tends to let one count dominate the population.
comparison.compare_length_distribution(results, normalize=True)

In [ ]:
# Best layout found at each sensor count, per strategy (table + grouped bars).
# Passing the evaluator adds a coverage % column (recomputes coverage per champion).
comparison.compare_best_per_length(results, evaluator=evaluator)

In [ ]:
# Per sensor count, both strategies overlaid. A count whose line breaks (e.g. a
# late gap) went extinct under that strategy; niching should keep every count's
# line continuous and improving.
comparison.compare_per_length_evolution(results, metric="max")